In [3]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

import warnings
warnings.simplefilter("ignore", FutureWarning)

import os
import pandas as pd
import numpy as np
from torchvision import transforms
from PIL import Image, UnidentifiedImageError
from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn as nn
import torchvision.models as models
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import roc_auc_score

#Full data
train_df_full = pd.read_csv("/mnt/Internal/MedImage/merged_dataset-Copy1.csv")

train_df_full = pd.get_dummies(train_df_full, columns=["GENDER", "PRIMARY_RACE", "ETHNICITY"])
train_df_full.columns.tolist()

# Convert categorical columns to integers in train_df_full

# Race
train_df_full['PRIMARY_RACE_American Indian or Alaska Native'] = train_df_full['PRIMARY_RACE_American Indian or Alaska Native'].astype(int)
train_df_full['PRIMARY_RACE_Asian'] = train_df_full['PRIMARY_RACE_Asian'].astype(int)
#train_df_full['PRIMARY_RACE_Asian - Historical Conv'] = train_df_full['PRIMARY_RACE_Asian - Historical Conv'].astype(int)
train_df_full['PRIMARY_RACE_Asian, Hispanic'] = train_df_full['PRIMARY_RACE_Asian, Hispanic'].astype(int)
train_df_full['PRIMARY_RACE_Asian, non-Hispanic'] = train_df_full['PRIMARY_RACE_Asian, non-Hispanic'].astype(int)
train_df_full['PRIMARY_RACE_Black or African American'] = train_df_full['PRIMARY_RACE_Black or African American'].astype(int)
train_df_full['PRIMARY_RACE_Black, Hispanic'] = train_df_full['PRIMARY_RACE_Black, Hispanic'].astype(int)
train_df_full['PRIMARY_RACE_Black, non-Hispanic'] = train_df_full['PRIMARY_RACE_Black, non-Hispanic'].astype(int)
train_df_full['PRIMARY_RACE_Native American, Hispanic'] = train_df_full['PRIMARY_RACE_Native American, Hispanic'].astype(int)
train_df_full['PRIMARY_RACE_Native American, non-Hispanic'] = train_df_full['PRIMARY_RACE_Native American, non-Hispanic'].astype(int)
train_df_full['PRIMARY_RACE_Native Hawaiian or Other Pacific Islander'] = train_df_full['PRIMARY_RACE_Native Hawaiian or Other Pacific Islander'].astype(int)
#train_df_full['PRIMARY_RACE_Other'] = train_df_full['PRIMARY_RACE_Other'].astype(int)
#train_df_full['PRIMARY_RACE_Other, Hispanic'] = train_df_full['PRIMARY_RACE_Other, Hispanic'].astype(int)
#train_df_full['PRIMARY_RACE_Other, non-Hispanic'] = train_df_full['PRIMARY_RACE_Other, non-Hispanic'].astype(int)
#train_df_full['PRIMARY_RACE_Pacific Islander, Hispanic'] = train_df_full['PRIMARY_RACE_Pacific Islander, Hispanic'].astype(int)
#train_df_full['PRIMARY_RACE_Pacific Islander, non-Hispanic'] = train_df_full['PRIMARY_RACE_Pacific Islander, non-Hispanic'].astype(int)
train_df_full['PRIMARY_RACE_Patient Refused'] = train_df_full['PRIMARY_RACE_Patient Refused'].astype(int)
train_df_full['PRIMARY_RACE_Race and Ethnicity Unknown'] = train_df_full['PRIMARY_RACE_Race and Ethnicity Unknown'].astype(int)
train_df_full['PRIMARY_RACE_Unknown'] = train_df_full['PRIMARY_RACE_Unknown'].astype(int)
train_df_full['PRIMARY_RACE_White'] = train_df_full['PRIMARY_RACE_White'].astype(int)
train_df_full['PRIMARY_RACE_White or Caucasian'] = train_df_full['PRIMARY_RACE_White or Caucasian'].astype(int)
train_df_full['PRIMARY_RACE_White, Hispanic'] = train_df_full['PRIMARY_RACE_White, Hispanic'].astype(int)
train_df_full['PRIMARY_RACE_White, non-Hispanic'] = train_df_full['PRIMARY_RACE_White, non-Hispanic'].astype(int)

# Ethnicity
#train_df_full['ETHNICITY_0'] = train_df_full['ETHNICITY_0'].astype(int)
#train_df_full['ETHNICITY_Hispanic'] = train_df_full['ETHNICITY_Hispanic'].astype(int)
train_df_full['ETHNICITY_Hispanic/Latino'] = train_df_full['ETHNICITY_Hispanic/Latino'].astype(int)
train_df_full['ETHNICITY_Non-Hispanic/Non-Latino'] = train_df_full['ETHNICITY_Non-Hispanic/Non-Latino'].astype(int)
train_df_full['ETHNICITY_Not Hispanic'] = train_df_full['ETHNICITY_Not Hispanic'].astype(int)
train_df_full['ETHNICITY_Patient Refused'] = train_df_full['ETHNICITY_Patient Refused'].astype(int)

# Gender
train_df_full['GENDER_Male'] = train_df_full['GENDER_Male'].astype(int)
train_df_full['GENDER_Female'] = train_df_full['GENDER_Female'].astype(int)

# Display the updated DataFrame
train_df_full.head()

train_df_full.replace(-1, 1, inplace=True)

train_df_full.replace(np.nan, 0, inplace=True)

# # Training Data
# train_df = train_df_full[1:170000]
# #Validation Data
# valid_df = train_df_full[170001:190000]

# Training Data
train_df = train_df_full[0:14000]
#Validation Data
valid_df = train_df_full[175000:180000]

train_image_root = "/mnt/Internal/MedImage/CheXpert Dataset/unzip_chexpert_images/CheXpert-v1.0/train"
valid_image_root = "/mnt/Internal/MedImage/CheXpert Dataset/unzip_chexpert_images/CheXpert-v1.0/train"


In [8]:
import torch
from torch.utils.data import DataLoader
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, roc_curve
import os

# ==== CONFIG ====
model_path = "/mnt/Internal/MedImage/CheXpert Dataset/Lab_Rotation_2/training_CNN_Models_On_new_diffusion_generated_data_using_generated_imgs_As_training/model_epoch_116.pth"
valid_csv = train_df_full
valid_image_root = "/mnt/Internal/MedImage/CheXpert Dataset/unzip_chexpert_images/CheXpert-v1.0/train"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
label_columns = ['No Finding', 'Enlarged Cardiomediastinum', 'Cardiomegaly', 'Lung Opacity',
                 'Lung Lesion', 'Edema', 'Consolidation', 'Pneumonia', 'Atelectasis',
                 'Pneumothorax', 'Pleural Effusion', 'Pleural Other', 'Fracture', 'Support Devices']
transform = None  # Add your transform if needed

# ==== LOAD DATAFRAME ====
valid_df = valid_df

# ==== LOAD MODEL ====
model = torchvision.models.densenet121(pretrained=False)  
num_ftrs = model.classifier.in_features  
model.classifier = nn.Linear(num_ftrs, len(disease_names))  
model.load_state_dict(torch.load(best_model_path, map_location=device))

# Move model to device
model.to(device)
model.eval()

print(f"Loaded model from {best_model_path}")
# ==== DEFINE DATASETS ====
male_df = valid_df[valid_df['GENDER_Male'] == 1]
female_df = valid_df[valid_df['GENDER_Female'] == 1]

male_dataset = CheXpertDataset(male_df, valid_image_root, transform=transform)
female_dataset = CheXpertDataset(female_df, valid_image_root, transform=transform)

male_loader = DataLoader(male_dataset, batch_size=2, shuffle=False, collate_fn=collate_fn)
female_loader = DataLoader(female_dataset, batch_size=2, shuffle=False, collate_fn=collate_fn)

# ==== EVALUATION UTILS ====
def get_probs_labels(model, loader):
    all_probs, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            if images.size(0) == 0:
                continue
            images = images.to(device)
            probs = torch.sigmoid(model(images)).cpu().numpy()
            all_probs.append(probs)
            all_labels.append(labels.numpy())
    return np.vstack(all_probs), np.vstack(all_labels)

def compute_auc(probs, labels, label_names):
    auc_dict = {}
    for i, name in enumerate(label_names):
        try:
            auc = roc_auc_score(labels[:, i], probs[:, i])
            auc_dict[name] = auc
        except ValueError:
            auc_dict[name] = np.nan
    return auc_dict

# ==== GET PROBS & LABELS ====
male_probs, male_labels = get_probs_labels(model, male_loader)
female_probs, female_labels = get_probs_labels(model, female_loader)

# ==== AUC SCORES ====
male_auc = compute_auc(male_probs, male_labels, label_columns)
female_auc = compute_auc(female_probs, female_labels, label_columns)

# ==== SUMMARY TABLE ====
summary_df = pd.DataFrame({
    "Disease": label_columns,
    "AUC_Male": [male_auc[l] for l in label_columns],
    "AUC_Female": [female_auc[l] for l in label_columns]
})
summary_df["AUC_Diff"] = summary_df["AUC_Male"] - summary_df["AUC_Female"]
print("\n=== AUC Summary ===")
print(summary_df.to_string(index=False))

# ==== PLOT ROC CURVES ====
output_dir = "roc_plots"
os.makedirs(output_dir, exist_ok=True)

for i, disease in enumerate(label_columns):
    try:
        fpr_male, tpr_male, _ = roc_curve(male_labels[:, i], male_probs[:, i])
        fpr_female, tpr_female, _ = roc_curve(female_labels[:, i], female_probs[:, i])
        
        plt.figure(figsize=(6, 5))
        plt.plot(fpr_male, tpr_male, label="Male ROC")
        plt.plot(fpr_female, tpr_female, label="Female ROC")
        plt.plot([0, 1], [0, 1], 'k--', label='Chance')
        plt.xlabel("False Positive Rate")
        plt.ylabel("True Positive Rate")
        plt.title(f"ROC Curve: {disease}")
        plt.legend()
        plt.grid(True)
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, f"roc_{disease.replace(' ', '_')}.png"))
        plt.close()
    except ValueError:
        print(f"ROC Curve skipped for {disease} due to missing data.")



NameError: name 'build_model' is not defined